## 05 Final Demo Checks

This notebook operationalizes the final overlap-detection configuration selected in Notebooks 03 and 04 and prepares the proof-of-concept demo artifacts.

### Main goal

The purpose of Notebook 05 is to verify that the final selected configuration works cleanly as an end-to-end demo setup.

It answers:

> Is the final selected overlap-detection configuration ready for practical proof-of-concept use?

### Final selected configuration

Based on Notebooks 03 and 04, the final selected configuration is:

- **Model:** OpenAI `text-embedding-3-large`
- **Text variant:** `embedding_text_enriched`
- **Similarity metric:** cosine
- **Locked threshold:** `0.660026`

### What this notebook does

- loads the final cleaned course dataset from Notebook 01
- rebuilds the final selected embedding input surface
- generates final embeddings for all courses using the selected model
- computes full pairwise cosine similarity across the course set
- applies the locked threshold to identify overlap candidates
- saves app-ready artifacts for the final Streamlit demo
- runs practical sanity checks on the final retrieval behavior

### What this notebook does not do

- it does not rerun the benchmark competition from Notebook 03
- it does not retune thresholds from Notebook 04
- it does not perform large-scale production deployment
- it does not replace a future pgvector-backed implementation

### Relationship to the final demo app

This notebook prepares and verifies the final artifacts that the Streamlit application can use.

In other words:

- **Notebook 05** = final system preparation and sanity checking
- **`app/streamlit_app.py`** = final interactive user-facing demo

In [2]:
# Cell 01 - Import libraries and define project paths

from pathlib import Path
import os
import time
import warnings
from itertools import combinations

import numpy as np
import pandas as pd

from dotenv import load_dotenv
from openai import OpenAI
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 160)

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
TABLES_DIR = REPORTS_DIR / "tables" / "final_demo"
FIGURES_DIR = REPORTS_DIR / "figures" / "final_demo"
ENV_PATH = PROJECT_ROOT / ".env"

COURSES_CSV_PATH = DATA_DIR / "courses_clean.csv"
FINAL_EMBEDDINGS_PARQUET_PATH = DATA_DIR / "final_embeddings.parquet"
FINAL_PAIRWISE_SIMILARITY_PARQUET_PATH = DATA_DIR / "final_pairwise_similarity.parquet"
FINAL_PAIRWISE_SIMILARITY_CSV_PATH = DATA_DIR / "final_pairwise_similarity.csv"
FINAL_OVERLAP_CANDIDATES_CSV_PATH = TABLES_DIR / "final_demo_overlap_candidates.csv"
FINAL_TOP_MATCHES_CSV_PATH = TABLES_DIR / "final_demo_top_matches_per_course.csv"
FINAL_SYSTEM_SUMMARY_CSV_PATH = TABLES_DIR / "final_demo_system_summary.csv"

for path in [TABLES_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

FINAL_MODEL_NAME = "openai_text-embedding-3-large"
FINAL_OPENAI_MODEL_ID = "text-embedding-3-large"
FINAL_TEXT_VARIANT = "embedding_text_enriched"
FINAL_SIMILARITY_METRIC = "cosine"
FINAL_LOCKED_THRESHOLD = 0.659769

print("Project root:", PROJECT_ROOT)
print("Courses CSV:", COURSES_CSV_PATH)
print("Final embeddings parquet:", FINAL_EMBEDDINGS_PARQUET_PATH)
print("Final pairwise similarity parquet:", FINAL_PAIRWISE_SIMILARITY_PARQUET_PATH)
print("Final overlap candidates CSV:", FINAL_OVERLAP_CANDIDATES_CSV_PATH)
print("Environment file:", ENV_PATH)
print("Final model:", FINAL_MODEL_NAME)
print("Final text variant:", FINAL_TEXT_VARIANT)
print("Final similarity metric:", FINAL_SIMILARITY_METRIC)
print("Final locked threshold:", FINAL_LOCKED_THRESHOLD)

Project root: C:\Users\user\Downloads\curriculum-overlap-poc
Courses CSV: C:\Users\user\Downloads\curriculum-overlap-poc\data\courses_clean.csv
Final embeddings parquet: C:\Users\user\Downloads\curriculum-overlap-poc\data\final_embeddings.parquet
Final pairwise similarity parquet: C:\Users\user\Downloads\curriculum-overlap-poc\data\final_pairwise_similarity.parquet
Final overlap candidates CSV: C:\Users\user\Downloads\curriculum-overlap-poc\reports\tables\final_demo\final_demo_overlap_candidates.csv
Environment file: C:\Users\user\Downloads\curriculum-overlap-poc\.env
Final model: openai_text-embedding-3-large
Final text variant: embedding_text_enriched
Final similarity metric: cosine
Final locked threshold: 0.659769


### Load environment variables and initialize the final embedding client

This notebook uses the final selected OpenAI embedding model.

Expected `.env` file location:

`curriculum-overlap-poc/.env`

Expected variable:

- `OPENAI_API_KEY`

In [3]:
# Cell 02 - Load environment variables and initialize OpenAI client

load_dotenv(ENV_PATH)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()

if not OPENAI_API_KEY:
    raise ValueError(
        "OPENAI_API_KEY was not found in the .env file. "
        "Notebook 05 requires the final selected OpenAI embedding model."
    )

openai_client = OpenAI(api_key=OPENAI_API_KEY)

print("OpenAI API key loaded: Yes")
print("OpenAI client initialized successfully.")

OpenAI API key loaded: Yes
OpenAI client initialized successfully.


### Load the cleaned course dataset

Notebook 05 uses the cleaned course dataset from Notebook 01 as the source of truth for final demo preparation.

In [4]:
# Cell 03 - Load cleaned course dataset

if not COURSES_CSV_PATH.exists():
    raise FileNotFoundError(f"Clean course dataset not found: {COURSES_CSV_PATH}")

courses_df = pd.read_csv(COURSES_CSV_PATH, encoding="utf-8", keep_default_na=False)

print("courses_df shape:", courses_df.shape)
print("Columns:")
print(courses_df.columns.tolist())

display(courses_df.head(3))

courses_df shape: (50, 25)
Columns:
['degree_programme', 'course_name', 'course_unit_code', 'credits', 'teaching_language_text', 'responsible_person_text', 'objective_text', 'competences_knowledge_text', 'competences_practice_text', 'competences_applying_technology_text', 'competences_multidisciplinary_text', 'competences_communication_text', 'competences_investigations_text', 'learning_outcomes_text', 'content_text', 'prerequisites_text', 'assessment_grading_scale', 'assessment_text', 'course_name_normalized', 'prerequisites_text_normalized', 'embedding_text_main', 'embedding_text_enriched', 'embedding_text_full', 'review_text_compact', 'review_text_full']


,degree_programme,course_name,course_unit_code,credits,teaching_language_text,responsible_person_text,objective_text,competences_knowledge_text,competences_practice_text,competences_applying_technology_text,competences_multidisciplinary_text,competences_communication_text,competences_investigations_text,learning_outcomes_text,content_text,prerequisites_text,assessment_grading_scale,assessment_text,course_name_normalized,prerequisites_text_normalized,embedding_text_main,embedding_text_enriched,embedding_text_full,review_text_compact,review_text_full
0,Information and Communication Technology,Data Networks,TT00CD70,5,"Finnish, English",Pasi Hyytiäinen,You learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data network. You are ...,You know the main modeling methods of data networks. You have knowledge and understanding of data networks and their effects. You know the basic principles ...,You can apply data network skills to communication between devices.,,,,,,"In this course, you will learn the principles of network structure and protocol design, as well as the principles of Internet communication. You will master...",Linux Basics,1-5,Grade 1: The student knows the theory on network protocols sufficiently and is able to recognize some sections of existing solutions. The student can design...,data networks,linux basics,Data Networks\nYou learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data ne...,Data Networks\nYou learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data ne...,Data Networks\nYou learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data ne...,Course code: TT00CD70\nCourse name: Data Networks\nObjective: You learn the structure and protocols of data networks used by computers and other end devices...,Course code: TT00CD70\nCourse name: Data Networks\nObjective: You learn the structure and protocols of data networks used by computers and other end devices...
1,Information and Communication Technology,Linux Basics,TT00CD71,4,"Finnish, English","Juho Pekki, Teemu Siikaniemi","After completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work with and manage t...",You understand the key concepts and architecture of the Linux operating system. You understand the importance of managing user accounts and groups.,You know how to manage the key functions of the Linux system and use command-line tools. You know how to manage the filesystem and user account permissions....,,,,,,"In this course, you will learn how to effectively manage the Linux operating system using a text-based interface. You will understand key concepts and be ab...",,1-5,"Grade 1: You are familiar with the basic concepts of the Linux operating system and can use basic commands. You know how to create user groups, add users to...",linux basics,,"Linux Basics\nAfter completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work wit...","Linux Basics\nAfter completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work wit...","Linux Basics\nAfter completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work wit...","Course code: TT00CD71\nCourse name: Linux Basics\nObjective: After completing the course, you will understand the most important concepts of the Linux opera...","Course code: TT00CD71\nCourse name: Linux Basics\nObjective: After completing the course, you will understand the most important concepts of the Linux opera..."
2,Information and Communication Techno

In [5]:
# Cell 04 - Validate required columns and final selected text variant

required_columns = [
    "course_unit_code",
    "course_name",
    "embedding_text_enriched",
]

missing_required_columns = [col for col in required_columns if col not in courses_df.columns]

if missing_required_columns:
    raise ValueError(f"Missing required columns in courses_df: {missing_required_columns}")

if courses_df["course_unit_code"].duplicated().any():
    raise ValueError("Duplicate course_unit_code values found in courses_df.")

empty_final_text_count = (courses_df["embedding_text_enriched"].fillna("").astype(str).str.strip() == "").sum()
if empty_final_text_count > 0:
    raise ValueError(
        f"Found {empty_final_text_count} empty rows in embedding_text_enriched. "
        "Notebook 05 requires a complete final text surface."
    )

print("All required columns are present.")
print("Unique course codes:", courses_df["course_unit_code"].nunique())
print("Empty embedding_text_enriched rows:", empty_final_text_count)

All required columns are present.
Unique course codes: 50
Empty embedding_text_enriched rows: 0


### Build the final course text dataframe

This section isolates the final selected embedding surface that will be used for the proof-of-concept demo.

In [6]:
# Cell 05 - Build the final course text dataframe

final_course_text_df = courses_df[[
    "course_unit_code",
    "course_name",
    "embedding_text_enriched",
]].copy()

final_course_text_df["embedding_text_enriched"] = (
    final_course_text_df["embedding_text_enriched"]
    .astype(str)
    .str.replace("\r\n", "\n", regex=False)
    .str.replace("\r", "\n", regex=False)
    .str.strip()
)

print("final_course_text_df shape:", final_course_text_df.shape)
display(final_course_text_df.head(3))

final_course_text_df shape: (50, 3)


,course_unit_code,course_name,embedding_text_enriched
0,TT00CD70,Data Networks,Data Networks\nYou learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data ne...
1,TT00CD71,Linux Basics,"Linux Basics\nAfter completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work wit..."
2,TT00CD72,Servers and containers,Servers and containers\nYou understand the fundamental types and functions of servers. You understand the key concepts of various services offered by cloud ...


### Helper functions for final embedding and similarity preparation

In [7]:
# Cell 06 - Define helper functions

def normalize_structured_text(text: str) -> str:
    text = str(text).replace("\r\n", "\n").replace("\r", "\n")
    lines = [line.strip() for line in text.split("\n")]
    lines = [line for line in lines if line]
    return "\n".join(lines)

def batched(iterable, batch_size: int):
    for start in range(0, len(iterable), batch_size):
        yield iterable[start:start + batch_size]

def l2_normalize_embeddings(matrix: np.ndarray) -> np.ndarray:
    return normalize(matrix, norm="l2")

def cosine_similarity_from_normalized(vec1: np.ndarray, vec2: np.ndarray) -> float:
    return float(np.dot(vec1, vec2))

def encode_with_openai(texts, model_name, client, batch_size=64):
    all_vectors = []

    for batch_texts in batched(texts, batch_size=batch_size):
        response = client.embeddings.create(
            model=model_name,
            input=batch_texts,
        )
        batch_vectors = [item.embedding for item in response.data]
        all_vectors.extend(batch_vectors)

    raw_matrix = np.array(all_vectors, dtype=np.float32)
    normalized_matrix = l2_normalize_embeddings(raw_matrix)

    return raw_matrix, normalized_matrix

### Generate final embeddings for all courses

This is the final selected embedding generation step for the proof-of-concept demo.

In [8]:
# Cell 07 - Generate final embeddings for all courses

final_texts = final_course_text_df["embedding_text_enriched"].apply(normalize_structured_text).tolist()

embedding_start_time = time.time()

raw_embedding_matrix, normalized_embedding_matrix = encode_with_openai(
    texts=final_texts,
    model_name=FINAL_OPENAI_MODEL_ID,
    client=openai_client,
    batch_size=64,
)

embedding_elapsed_seconds = time.time() - embedding_start_time

if raw_embedding_matrix.shape[0] != len(final_course_text_df):
    raise ValueError("Raw embedding row count does not match course count.")

if normalized_embedding_matrix.shape[0] != len(final_course_text_df):
    raise ValueError("Normalized embedding row count does not match course count.")

print("Raw embedding matrix shape:", raw_embedding_matrix.shape)
print("Normalized embedding matrix shape:", normalized_embedding_matrix.shape)
print(f"Embedding generation time: {embedding_elapsed_seconds:.2f} seconds")

Raw embedding matrix shape: (50, 3072)
Normalized embedding matrix shape: (50, 3072)
Embedding generation time: 3.62 seconds


In [9]:
# Cell 08 - Audit final embedding norms

raw_embedding_norms = np.linalg.norm(raw_embedding_matrix, axis=1)
normalized_embedding_norms = np.linalg.norm(normalized_embedding_matrix, axis=1)

embedding_norm_summary_df = pd.DataFrame([
    {
        "matrix_type": "raw",
        "norm_mean": float(raw_embedding_norms.mean()),
        "norm_std": float(raw_embedding_norms.std()),
        "norm_min": float(raw_embedding_norms.min()),
        "norm_max": float(raw_embedding_norms.max()),
    },
    {
        "matrix_type": "normalized",
        "norm_mean": float(normalized_embedding_norms.mean()),
        "norm_std": float(normalized_embedding_norms.std()),
        "norm_min": float(normalized_embedding_norms.min()),
        "norm_max": float(normalized_embedding_norms.max()),
    },
])

display(embedding_norm_summary_df)

,matrix_type,norm_mean,norm_std,norm_min,norm_max
0,raw,0.999973,2.354011e-04,0.999485,1.000512
1,normalized,1.000000,1.486536e-07,0.999999,1.000000


### Save the final embedding artifact

This artifact is intended to be reusable by the final Streamlit application and by later pgvector preparation work.

In [10]:
# Cell 09 - Save the final embedding artifact

final_embeddings_df = final_course_text_df.copy()

final_embeddings_df["final_model_name"] = FINAL_MODEL_NAME
final_embeddings_df["final_openai_model_id"] = FINAL_OPENAI_MODEL_ID
final_embeddings_df["final_text_variant"] = FINAL_TEXT_VARIANT
final_embeddings_df["final_similarity_metric"] = FINAL_SIMILARITY_METRIC
final_embeddings_df["final_locked_threshold"] = FINAL_LOCKED_THRESHOLD
final_embeddings_df["embedding_dim"] = normalized_embedding_matrix.shape[1]
final_embeddings_df["embedding_generation_seconds"] = embedding_elapsed_seconds
final_embeddings_df["raw_embedding_norm"] = raw_embedding_norms.astype(float)
final_embeddings_df["normalized_embedding_norm"] = normalized_embedding_norms.astype(float)
final_embeddings_df["embedding_vector"] = [vec.tolist() for vec in normalized_embedding_matrix]

final_embeddings_df.to_parquet(FINAL_EMBEDDINGS_PARQUET_PATH, index=False)

print("Saved final embedding artifact to:", FINAL_EMBEDDINGS_PARQUET_PATH)
print("final_embeddings_df shape:", final_embeddings_df.shape)

display(final_embeddings_df.head(3))

Saved final embedding artifact to: C:\Users\user\Downloads\curriculum-overlap-poc\data\final_embeddings.parquet
final_embeddings_df shape: (50, 13)


,course_unit_code,course_name,embedding_text_enriched,final_model_name,final_openai_model_id,final_text_variant,final_similarity_metric,final_locked_threshold,embedding_dim,embedding_generation_seconds,raw_embedding_norm,normalized_embedding_norm,embedding_vector
0,TT00CD70,Data Networks,Data Networks\nYou learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data ne...,openai_text-embedding-3-large,text-embedding-3-large,embedding_text_enriched,cosine,0.659769,3072,3.619459,1.000048,1.0,"[-0.02810533344745636, 0.020735694095492363, -0.005145777948200703, 0.017394179478287697, 0.019438758492469788, -9.685286931926385e-05, -0.02070517651736736..."
1,TT00CD71,Linux Basics,"Linux Basics\nAfter completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work wit...",openai_text-embedding-3-large,text-embedding-3-large,embedding_text_enriched,cosine,0.659769,3072,3.619459,0.999847,1.0,"[-0.013971555046737194, 0.03848854452371597, -0.012353876605629921, 0.0024856547825038433, -0.017489243298768997, 0.0045134760439395905, -0.0257302466779947..."
2,TT00CD72,Servers and containers,Servers and containers\nYou understand the fundamental types and functions of servers. You understand the key concepts of various services offered by cloud ...,openai_text-embedding-3-large,text-embedding-3-large,embedding_text_enriched,cosine,0.659769,3072,3.619459,0.999609,1.0,"[-0.028056597337126732, 0.029399896040558815, -0.013387179933488369, 0.038039740175008774, 0.031140075996518135, -0.05470883846282959, -0.014768638648092747..."


### Compute full pairwise similarity across the course set

This section creates the final similarity surface used for sanity checks and app-readiness validation.

In [11]:
# Cell 10 - Compute full pairwise similarity across the course set

course_code_order = sorted(final_course_text_df["course_unit_code"].tolist())
course_name_map = dict(zip(final_course_text_df["course_unit_code"], final_course_text_df["course_name"]))
code_to_index = {
    code: idx for idx, code in enumerate(final_course_text_df["course_unit_code"].tolist())
}

pair_records = []

for code_1, code_2 in combinations(course_code_order, 2):
    idx1 = code_to_index[code_1]
    idx2 = code_to_index[code_2]

    vec1 = normalized_embedding_matrix[idx1]
    vec2 = normalized_embedding_matrix[idx2]

    similarity_score = cosine_similarity_from_normalized(vec1, vec2)
    overlap_flag = int(similarity_score >= FINAL_LOCKED_THRESHOLD)

    left_code, right_code = sorted([code_1, code_2])

    pair_records.append({
        "pair_id": f"{left_code}__{right_code}",
        "course_unit_code_1": left_code,
        "course_name_1": course_name_map[left_code],
        "course_unit_code_2": right_code,
        "course_name_2": course_name_map[right_code],
        "similarity_score": similarity_score,
        "overlap_flag": overlap_flag,
        "final_locked_threshold": FINAL_LOCKED_THRESHOLD,
        "final_model_name": FINAL_MODEL_NAME,
        "final_text_variant": FINAL_TEXT_VARIANT,
        "final_similarity_metric": FINAL_SIMILARITY_METRIC,
    })

final_pairwise_similarity_df = pd.DataFrame(pair_records)

expected_pair_count = len(course_code_order) * (len(course_code_order) - 1) // 2
if len(final_pairwise_similarity_df) != expected_pair_count:
    raise ValueError("Final pairwise similarity row count does not match expected pair count.")

print("Expected pair count:", expected_pair_count)
print("Actual pair count:", len(final_pairwise_similarity_df))
display(final_pairwise_similarity_df.head(5))

Expected pair count: 1225
Actual pair count: 1225


,pair_id,course_unit_code_1,course_name_1,course_unit_code_2,course_name_2,similarity_score,overlap_flag,final_locked_threshold,final_model_name,final_text_variant,final_similarity_metric
0,TT00CD70__TT00CD71,TT00CD70,Data Networks,TT00CD71,Linux Basics,0.502033,0,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
1,TT00CD70__TT00CD72,TT00CD70,Data Networks,TT00CD72,Servers and containers,0.561306,0,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
2,TT00CD70__TT00CD73,TT00CD70,Data Networks,TT00CD73,Digital Technology,0.530214,0,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
3,TT00CD70__TT00CD74,TT00CD70,Data Networks,TT00CD74,Information Systems and Architecture,0.517068,0,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
4,TT00CD70__TT00CD75,TT00CD70,Data Networks,TT00CD75,Windows Basics,0.496574,0,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine


In [13]:
# Cell 11 - Save full pairwise similarity artifact

final_pairwise_similarity_df.to_parquet(FINAL_PAIRWISE_SIMILARITY_PARQUET_PATH, index=False)
final_pairwise_similarity_df.to_csv(FINAL_PAIRWISE_SIMILARITY_CSV_PATH, index=False, encoding="utf-8")

print("Saved pairwise similarity parquet to:", FINAL_PAIRWISE_SIMILARITY_PARQUET_PATH)
print("Saved pairwise similarity CSV to:", FINAL_PAIRWISE_SIMILARITY_CSV_PATH)

Saved pairwise similarity parquet to: C:\Users\user\Downloads\curriculum-overlap-poc\data\final_pairwise_similarity.parquet
Saved pairwise similarity CSV to: C:\Users\user\Downloads\curriculum-overlap-poc\data\final_pairwise_similarity.csv


### Build final overlap candidates and top-match views

These outputs are intended to be directly useful for the final proof-of-concept application.

In [14]:
# Cell 12 - Build final overlap candidate table

final_overlap_candidates_df = final_pairwise_similarity_df[
    final_pairwise_similarity_df["overlap_flag"] == 1
].copy()

final_overlap_candidates_df = final_overlap_candidates_df.sort_values(
    ["similarity_score", "pair_id"],
    ascending=[False, True]
).reset_index(drop=True)

final_overlap_candidates_df.to_csv(FINAL_OVERLAP_CANDIDATES_CSV_PATH, index=False, encoding="utf-8")

print("Saved final overlap candidates to:", FINAL_OVERLAP_CANDIDATES_CSV_PATH)
print("Number of above-threshold overlap candidates:", len(final_overlap_candidates_df))

display(final_overlap_candidates_df.head(20))

Saved final overlap candidates to: C:\Users\user\Downloads\curriculum-overlap-poc\reports\tables\final_demo\final_demo_overlap_candidates.csv
Number of above-threshold overlap candidates: 28


,pair_id,course_unit_code_1,course_name_1,course_unit_code_2,course_name_2,similarity_score,overlap_flag,final_locked_threshold,final_model_name,final_text_variant,final_similarity_metric
0,TT00CD92__TT00CD94,TT00CD92,Mobile Project,TT00CD94,Mobile Application Development,0.830978,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
1,TT00CD93__TT00CD94,TT00CD93,Android Application Development,TT00CD94,Mobile Application Development,0.784151,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
2,TT00CE05__TT00CE06,TT00CE05,Data Analytics Project,TT00CE06,Machine Learning Project,0.778475,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
3,TT00CD87__TT00CE11,TT00CD87,Information Security Technologies,TT00CE11,Data Security Controls,0.748164,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
4,TT00CD75__TT00CE25,TT00CD75,Windows Basics,TT00CE25,Windows Infrastructure,0.738116,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
5,TT00CD86__TT00CE08,TT00CD86,Cyber Security,TT00CE08,Basics of Cyber Security Exercises,0.734373,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
6,TT00CD86__TT00CE11,TT00CD86,Cyber Security,TT00CE11,Data Security Controls,0.731628,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
7,TT00CD82__TT00CD89,TT00CD82,Project Management and Quality Assurance,TT00CD89,Software Testing,0.731367,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
8,TT00CD96__TT00CK80,TT00CD96,Graphics Programming,TT00CK80,Mathematical Basics of Graphics Programming,0.722048,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
9,TT00CD89__TT00CE09,TT00CD89,Software Testing,TT00CE09,Auditing and Penetration Testing,0.719062,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine


In [15]:
# Cell 13 - Build top matches per course for demo inspection

top_match_rows = []

for source_code in course_code_order:
    source_name = course_name_map[source_code]

    left_df = final_pairwise_similarity_df[
        final_pairwise_similarity_df["course_unit_code_1"] == source_code
    ][[
        "course_unit_code_1",
        "course_name_1",
        "course_unit_code_2",
        "course_name_2",
        "similarity_score",
        "overlap_flag",
    ]].copy()

    left_df = left_df.rename(columns={
        "course_unit_code_1": "source_course_unit_code",
        "course_name_1": "source_course_name",
        "course_unit_code_2": "matched_course_unit_code",
        "course_name_2": "matched_course_name",
    })

    right_df = final_pairwise_similarity_df[
        final_pairwise_similarity_df["course_unit_code_2"] == source_code
    ][[
        "course_unit_code_2",
        "course_name_2",
        "course_unit_code_1",
        "course_name_1",
        "similarity_score",
        "overlap_flag",
    ]].copy()

    right_df = right_df.rename(columns={
        "course_unit_code_2": "source_course_unit_code",
        "course_name_2": "source_course_name",
        "course_unit_code_1": "matched_course_unit_code",
        "course_name_1": "matched_course_name",
    })

    source_matches_df = pd.concat([left_df, right_df], ignore_index=True)
    source_matches_df = source_matches_df.sort_values(
        ["similarity_score", "matched_course_unit_code"],
        ascending=[False, True]
    ).reset_index(drop=True)

    source_matches_df["match_rank"] = range(1, len(source_matches_df) + 1)
    source_matches_df["final_locked_threshold"] = FINAL_LOCKED_THRESHOLD

    top_match_rows.append(source_matches_df.head(10))

final_top_matches_df = pd.concat(top_match_rows, ignore_index=True)
final_top_matches_df.to_csv(FINAL_TOP_MATCHES_CSV_PATH, index=False, encoding="utf-8")

print("Saved top matches per course to:", FINAL_TOP_MATCHES_CSV_PATH)
print("final_top_matches_df shape:", final_top_matches_df.shape)

display(final_top_matches_df.head(20))

Saved top matches per course to: C:\Users\user\Downloads\curriculum-overlap-poc\reports\tables\final_demo\final_demo_top_matches_per_course.csv
final_top_matches_df shape: (500, 8)


,source_course_unit_code,source_course_name,matched_course_unit_code,matched_course_name,similarity_score,overlap_flag,match_rank,final_locked_threshold
0,TT00CD70,Data Networks,TT00CE21,Cloud Architectures and Platforms,0.592988,0,1,0.659769
1,TT00CD70,Data Networks,TT00CD86,Cyber Security,0.578102,0,2,0.659769
2,TT00CD70,Data Networks,TT00CE11,Data Security Controls,0.572231,0,3,0.659769
3,TT00CD70,Data Networks,TT00CD87,Information Security Technologies,0.564286,0,4,0.659769
4,TT00CD70,Data Networks,TT00CE25,Windows Infrastructure,0.561348,0,5,0.659769
5,TT00CD70,Data Networks,TT00CD72,Servers and containers,0.561306,0,6,0.659769
6,TT00CD70,Data Networks,TT00CD77,Basics of Programming,0.538729,0,7,0.659769
7,TT00CD70,Data Networks,TT00CD90,Software Design,0.531422,0,8,0.659769
8,TT00CD70,Data Networks,TT00CD73,Digital Technology,0.530214,0,9,0.659769
9,TT00CD70,Data Networks,TT00CD88,Hardening,0.529083,0,10,0.659769


In [16]:
# Cell 14 - Build final system summary

final_system_summary_df = pd.DataFrame([
    {"metric": "course_count", "value": int(len(final_course_text_df))},
    {"metric": "embedding_dimension", "value": int(normalized_embedding_matrix.shape[1])},
    {"metric": "pair_count", "value": int(len(final_pairwise_similarity_df))},
    {"metric": "above_threshold_pair_count", "value": int(final_pairwise_similarity_df["overlap_flag"].sum())},
    {"metric": "locked_threshold", "value": float(FINAL_LOCKED_THRESHOLD)},
    {"metric": "embedding_generation_seconds", "value": float(embedding_elapsed_seconds)},
    {"metric": "mean_similarity", "value": float(final_pairwise_similarity_df["similarity_score"].mean())},
    {"metric": "median_similarity", "value": float(final_pairwise_similarity_df["similarity_score"].median())},
    {"metric": "min_similarity", "value": float(final_pairwise_similarity_df["similarity_score"].min())},
    {"metric": "max_similarity", "value": float(final_pairwise_similarity_df["similarity_score"].max())},
])

final_system_summary_df.to_csv(FINAL_SYSTEM_SUMMARY_CSV_PATH, index=False, encoding="utf-8")

print("Saved final system summary to:", FINAL_SYSTEM_SUMMARY_CSV_PATH)
display(final_system_summary_df)

Saved final system summary to: C:\Users\user\Downloads\curriculum-overlap-poc\reports\tables\final_demo\final_demo_system_summary.csv


,metric,value
0,course_count,50.000000
1,embedding_dimension,3072.000000
2,pair_count,1225.000000
3,above_threshold_pair_count,28.000000
4,locked_threshold,0.659769
5,embedding_generation_seconds,3.619459
6,mean_similarity,0.462935
7,median_similarity,0.449255
8,min_similarity,0.280686
9,max_similarity,0.830978


### Initial final-demo sanity checks

Before finishing the notebook, inspect whether the final selected configuration behaves sensibly as a reusable proof-of-concept system.

In [17]:
# Cell 15 - Initial final-demo sanity checks

print("Top 15 above-threshold overlap candidates:")
display(final_overlap_candidates_df.head(15))

print()
print("Top 15 strongest course-to-course matches overall:")
display(
    final_pairwise_similarity_df
    .sort_values(["similarity_score", "pair_id"], ascending=[False, True])
    .head(15)
)

print()
print("Top 20 matches for one example course:")
example_course_code = final_course_text_df.iloc[0]["course_unit_code"]
example_matches_df = final_top_matches_df[
    final_top_matches_df["source_course_unit_code"] == example_course_code
].copy()

print("Example source course:", example_course_code, "|", course_name_map[example_course_code])
display(example_matches_df.head(20))

Top 15 above-threshold overlap candidates:


,pair_id,course_unit_code_1,course_name_1,course_unit_code_2,course_name_2,similarity_score,overlap_flag,final_locked_threshold,final_model_name,final_text_variant,final_similarity_metric
0,TT00CD92__TT00CD94,TT00CD92,Mobile Project,TT00CD94,Mobile Application Development,0.830978,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
1,TT00CD93__TT00CD94,TT00CD93,Android Application Development,TT00CD94,Mobile Application Development,0.784151,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
2,TT00CE05__TT00CE06,TT00CE05,Data Analytics Project,TT00CE06,Machine Learning Project,0.778475,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
3,TT00CD87__TT00CE11,TT00CD87,Information Security Technologies,TT00CE11,Data Security Controls,0.748164,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
4,TT00CD75__TT00CE25,TT00CD75,Windows Basics,TT00CE25,Windows Infrastructure,0.738116,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
5,TT00CD86__TT00CE08,TT00CD86,Cyber Security,TT00CE08,Basics of Cyber Security Exercises,0.734373,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
6,TT00CD86__TT00CE11,TT00CD86,Cyber Security,TT00CE11,Data Security Controls,0.731628,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
7,TT00CD82__TT00CD89,TT00CD82,Project Management and Quality Assurance,TT00CD89,Software Testing,0.731367,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
8,TT00CD96__TT00CK80,TT00CD96,Graphics Programming,TT00CK80,Mathematical Basics of Graphics Programming,0.722048,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
9,TT00CD89__TT00CE09,TT00CD89,Software Testing,TT00CE09,Auditing and Penetration Testing,0.719062,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine



Top 15 strongest course-to-course matches overall:


,pair_id,course_unit_code_1,course_name_1,course_unit_code_2,course_name_2,similarity_score,overlap_flag,final_locked_threshold,final_model_name,final_text_variant,final_similarity_metric
848,TT00CD92__TT00CD94,TT00CD92,Mobile Project,TT00CD94,Mobile Application Development,0.830978,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
874,TT00CD93__TT00CD94,TT00CD93,Android Application Development,TT00CD94,Mobile Application Development,0.784151,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
1120,TT00CE05__TT00CE06,TT00CE05,Data Analytics Project,TT00CE06,Machine Learning Project,0.778475,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
720,TT00CD87__TT00CE11,TT00CD87,Information Security Technologies,TT00CE11,Data Security Controls,0.748164,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
277,TT00CD75__TT00CE25,TT00CD75,Windows Basics,TT00CE25,Windows Infrastructure,0.738116,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
685,TT00CD86__TT00CE08,TT00CD86,Cyber Security,TT00CE08,Basics of Cyber Security Exercises,0.734373,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
688,TT00CD86__TT00CE11,TT00CD86,Cyber Security,TT00CE11,Data Security Controls,0.731628,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
528,TT00CD82__TT00CD89,TT00CD82,Project Management and Quality Assurance,TT00CD89,Software Testing,0.731367,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
971,TT00CD96__TT00CK80,TT00CD96,Graphics Programming,TT00CK80,Mathematical Basics of Graphics Programming,0.722048,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine
779,TT00CD89__TT00CE09,TT00CD89,Software Testing,TT00CE09,Auditing and Penetration Testing,0.719062,1,0.659769,openai_text-embedding-3-large,embedding_text_enriched,cosine



Top 20 matches for one example course:
Example source course: TT00CD70 | Data Networks


,source_course_unit_code,source_course_name,matched_course_unit_code,matched_course_name,similarity_score,overlap_flag,match_rank,final_locked_threshold
0,TT00CD70,Data Networks,TT00CE21,Cloud Architectures and Platforms,0.592988,0,1,0.659769
1,TT00CD70,Data Networks,TT00CD86,Cyber Security,0.578102,0,2,0.659769
2,TT00CD70,Data Networks,TT00CE11,Data Security Controls,0.572231,0,3,0.659769
3,TT00CD70,Data Networks,TT00CD87,Information Security Technologies,0.564286,0,4,0.659769
4,TT00CD70,Data Networks,TT00CE25,Windows Infrastructure,0.561348,0,5,0.659769
5,TT00CD70,Data Networks,TT00CD72,Servers and containers,0.561306,0,6,0.659769
6,TT00CD70,Data Networks,TT00CD77,Basics of Programming,0.538729,0,7,0.659769
7,TT00CD70,Data Networks,TT00CD90,Software Design,0.531422,0,8,0.659769
8,TT00CD70,Data Networks,TT00CD73,Digital Technology,0.530214,0,9,0.659769
9,TT00CD70,Data Networks,TT00CD88,Hardening,0.529083,0,10,0.659769


### Final demo sanity checks and app-readiness interpretation

This final section checks whether the generated artifacts are internally consistent and whether the final selected setup is suitable for use in the proof-of-concept application.

The goal is not to claim perfect production calibration.  
The goal is to confirm that the final selected configuration produces a stable, interpretable, and reusable overlap-detection surface for the demo.

In [18]:
# Cell 16 - Final artifact integrity checks

artifact_integrity_summary = {
    "course_count_final_embeddings": len(final_embeddings_df),
    "unique_course_codes_final_embeddings": final_embeddings_df["course_unit_code"].nunique(),
    "empty_embedding_vector_rows": final_embeddings_df["embedding_vector"].isna().sum(),
    "pair_count_final_pairwise_similarity": len(final_pairwise_similarity_df),
    "unique_pair_ids_final_pairwise_similarity": final_pairwise_similarity_df["pair_id"].nunique(),
    "duplicate_pair_ids_final_pairwise_similarity": final_pairwise_similarity_df["pair_id"].duplicated().sum(),
    "top_matches_row_count": len(final_top_matches_df),
    "overlap_candidate_count": len(final_overlap_candidates_df),
}

artifact_integrity_series = pd.Series(artifact_integrity_summary)

display(artifact_integrity_series)

if artifact_integrity_summary["course_count_final_embeddings"] != 50:
    raise ValueError("Final embedding artifact does not contain the expected 50 courses.")

if artifact_integrity_summary["unique_course_codes_final_embeddings"] != 50:
    raise ValueError("Final embedding artifact does not contain 50 unique course codes.")

if artifact_integrity_summary["empty_embedding_vector_rows"] > 0:
    raise ValueError("Some final embedding rows are missing embedding vectors.")

if artifact_integrity_summary["pair_count_final_pairwise_similarity"] != 1225:
    raise ValueError("Final pairwise similarity artifact does not contain the expected 1225 pairs.")

if artifact_integrity_summary["duplicate_pair_ids_final_pairwise_similarity"] > 0:
    raise ValueError("Duplicate pair_id values found in final pairwise similarity artifact.")

course_count_final_embeddings                     50
unique_course_codes_final_embeddings              50
empty_embedding_vector_rows                        0
pair_count_final_pairwise_similarity            1225
unique_pair_ids_final_pairwise_similarity       1225
duplicate_pair_ids_final_pairwise_similarity       0
top_matches_row_count                            500
overlap_candidate_count                           28
dtype: int64

In [19]:
# Cell 17 - Distribution checks for final similarity scores

similarity_distribution_df = pd.DataFrame([
    {
        "metric": "mean_similarity",
        "value": float(final_pairwise_similarity_df["similarity_score"].mean()),
    },
    {
        "metric": "median_similarity",
        "value": float(final_pairwise_similarity_df["similarity_score"].median()),
    },
    {
        "metric": "std_similarity",
        "value": float(final_pairwise_similarity_df["similarity_score"].std()),
    },
    {
        "metric": "min_similarity",
        "value": float(final_pairwise_similarity_df["similarity_score"].min()),
    },
    {
        "metric": "p05_similarity",
        "value": float(final_pairwise_similarity_df["similarity_score"].quantile(0.05)),
    },
    {
        "metric": "p25_similarity",
        "value": float(final_pairwise_similarity_df["similarity_score"].quantile(0.25)),
    },
    {
        "metric": "p50_similarity",
        "value": float(final_pairwise_similarity_df["similarity_score"].quantile(0.50)),
    },
    {
        "metric": "p75_similarity",
        "value": float(final_pairwise_similarity_df["similarity_score"].quantile(0.75)),
    },
    {
        "metric": "p95_similarity",
        "value": float(final_pairwise_similarity_df["similarity_score"].quantile(0.95)),
    },
    {
        "metric": "max_similarity",
        "value": float(final_pairwise_similarity_df["similarity_score"].max()),
    },
])

display(similarity_distribution_df)

,metric,value
0,mean_similarity,0.462935
1,median_similarity,0.449255
2,std_similarity,0.079678
3,min_similarity,0.280686
4,p05_similarity,0.355126
5,p25_similarity,0.409011
6,p50_similarity,0.449255
7,p75_similarity,0.503271
8,p95_similarity,0.617638
9,max_similarity,0.830978


In [20]:
# Cell 18 - Inspect course-level above-threshold match counts

course_overlap_count_rows = []

for source_code in course_code_order:
    source_matches_df = final_top_matches_df[
        final_top_matches_df["source_course_unit_code"] == source_code
    ].copy()

    all_source_matches_left = final_pairwise_similarity_df[
        final_pairwise_similarity_df["course_unit_code_1"] == source_code
    ][["similarity_score", "overlap_flag"]].copy()

    all_source_matches_right = final_pairwise_similarity_df[
        final_pairwise_similarity_df["course_unit_code_2"] == source_code
    ][["similarity_score", "overlap_flag"]].copy()

    all_source_matches_df = pd.concat(
        [all_source_matches_left, all_source_matches_right],
        ignore_index=True
    )

    course_overlap_count_rows.append({
        "course_unit_code": source_code,
        "course_name": course_name_map[source_code],
        "above_threshold_match_count": int(all_source_matches_df["overlap_flag"].sum()),
        "top10_above_threshold_match_count": int(source_matches_df["overlap_flag"].sum()),
        "best_similarity_score": float(all_source_matches_df["similarity_score"].max()),
        "mean_similarity_score": float(all_source_matches_df["similarity_score"].mean()),
    })

course_overlap_count_df = pd.DataFrame(course_overlap_count_rows).sort_values(
    ["above_threshold_match_count", "best_similarity_score", "course_unit_code"],
    ascending=[False, False, True]
).reset_index(drop=True)

display(course_overlap_count_df.head(20))

,course_unit_code,course_name,above_threshold_match_count,top10_above_threshold_match_count,best_similarity_score,mean_similarity_score
0,TT00CD87,Information Security Technologies,4,4,0.748164,0.506527
1,TT00CD86,Cyber Security,4,4,0.734373,0.524884
2,TT00CE11,Data Security Controls,3,3,0.748164,0.506768
3,TT00CD89,Software Testing,3,3,0.731367,0.502825
4,TT00CE09,Auditing and Penetration Testing,3,3,0.719062,0.473698
5,TT00CD92,Mobile Project,2,2,0.830978,0.465992
6,TT00CD94,Mobile Application Development,2,2,0.830978,0.483603
7,TT00CD93,Android Application Development,2,2,0.784151,0.467913
8,TT00CE05,Data Analytics Project,2,2,0.778475,0.449101
9,TT00CE06,Machine Learning Project,2,2,0.778475,0.467496


In [21]:
# Cell 19 - Save a pgvector-ready export table

pgvector_ready_df = final_embeddings_df[[
    "course_unit_code",
    "course_name",
    "embedding_text_enriched",
    "final_model_name",
    "final_openai_model_id",
    "final_text_variant",
    "final_similarity_metric",
    "final_locked_threshold",
    "embedding_dim",
    "embedding_vector",
]].copy()

PGVECTOR_READY_PARQUET_PATH = DATA_DIR / "pgvector_ready_embeddings.parquet"
PGVECTOR_READY_CSV_PATH = DATA_DIR / "pgvector_ready_embeddings.csv"

pgvector_ready_df.to_parquet(PGVECTOR_READY_PARQUET_PATH, index=False)
pgvector_ready_df.to_csv(PGVECTOR_READY_CSV_PATH, index=False, encoding="utf-8")

print("Saved pgvector-ready parquet to:", PGVECTOR_READY_PARQUET_PATH)
print("Saved pgvector-ready CSV to:", PGVECTOR_READY_CSV_PATH)
print("pgvector_ready_df shape:", pgvector_ready_df.shape)

display(pgvector_ready_df.head(3))

Saved pgvector-ready parquet to: C:\Users\user\Downloads\curriculum-overlap-poc\data\pgvector_ready_embeddings.parquet
Saved pgvector-ready CSV to: C:\Users\user\Downloads\curriculum-overlap-poc\data\pgvector_ready_embeddings.csv
pgvector_ready_df shape: (50, 10)


,course_unit_code,course_name,embedding_text_enriched,final_model_name,final_openai_model_id,final_text_variant,final_similarity_metric,final_locked_threshold,embedding_dim,embedding_vector
0,TT00CD70,Data Networks,Data Networks\nYou learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data ne...,openai_text-embedding-3-large,text-embedding-3-large,embedding_text_enriched,cosine,0.659769,3072,"[-0.02810533344745636, 0.020735694095492363, -0.005145777948200703, 0.017394179478287697, 0.019438758492469788, -9.685286931926385e-05, -0.02070517651736736..."
1,TT00CD71,Linux Basics,"Linux Basics\nAfter completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work wit...",openai_text-embedding-3-large,text-embedding-3-large,embedding_text_enriched,cosine,0.659769,3072,"[-0.013971555046737194, 0.03848854452371597, -0.012353876605629921, 0.0024856547825038433, -0.017489243298768997, 0.0045134760439395905, -0.0257302466779947..."
2,TT00CD72,Servers and containers,Servers and containers\nYou understand the fundamental types and functions of servers. You understand the key concepts of various services offered by cloud ...,openai_text-embedding-3-large,text-embedding-3-large,embedding_text_enriched,cosine,0.659769,3072,"[-0.028056597337126732, 0.029399896040558815, -0.013387179933488369, 0.038039740175008774, 0.031140075996518135, -0.05470883846282959, -0.014768638648092747..."


### Final interpretation for the proof-of-concept demo

The final selected configuration produces a stable and interpretable retrieval surface for the JAMK ICT course set.

Key observations from this notebook:

- all 50 courses were embedded successfully with the final selected OpenAI model
- the full pairwise similarity surface was generated successfully for all 1225 course pairs
- the locked threshold produced a small and manageable set of overlap candidates
- the strongest retrieved pairs are semantically plausible and suitable for human inspection
- the final artifacts are now available in app-ready and pgvector-ready form

This does not guarantee perfect production behavior.  
However, it does show that the selected configuration is practically usable for the proof-of-concept application and suitable for a final thesis demo.

### Files produced by this notebook

Main artifacts:

- `data/final_embeddings.parquet`
- `data/final_pairwise_similarity.parquet`
- `data/final_pairwise_similarity.csv`
- `data/pgvector_ready_embeddings.parquet`
- `data/pgvector_ready_embeddings.csv`

Demo support tables:

- `reports/tables/final_demo/final_demo_overlap_candidates.csv`
- `reports/tables/final_demo/final_demo_top_matches_per_course.csv`
- `reports/tables/final_demo/final_demo_system_summary.csv`

### Final role of Notebook 05

Notebook 05 does not replace the final Streamlit application.  
Instead, it verifies and operationalizes the final selected overlap-detection configuration so that the application can use a clean, validated, and reusable set of artifacts.

In [22]:
# Cell 20 - Final locked summary printout

print("Notebook 05 final status:")
print()
print("Final selected configuration")
print(f"  Model: {FINAL_MODEL_NAME}")
print(f"  Text variant: {FINAL_TEXT_VARIANT}")
print(f"  Similarity metric: {FINAL_SIMILARITY_METRIC}")
print(f"  Locked threshold: {FINAL_LOCKED_THRESHOLD}")
print()
print("Final system outputs")
print(f"  Courses embedded: {len(final_embeddings_df)}")
print(f"  Pairwise similarities computed: {len(final_pairwise_similarity_df)}")
print(f"  Above-threshold overlap candidates: {len(final_overlap_candidates_df)}")
print()
print("App-readiness")
print("  Final embedding artifact saved.")
print("  Final pairwise similarity artifact saved.")
print("  Final top-match table saved.")
print("  pgvector-ready export saved.")
print()
print("Notebook 05 is complete and the project is ready to move into the Streamlit demo app stage.")

Notebook 05 final status:

Final selected configuration
  Model: openai_text-embedding-3-large
  Text variant: embedding_text_enriched
  Similarity metric: cosine
  Locked threshold: 0.659769

Final system outputs
  Courses embedded: 50
  Pairwise similarities computed: 1225
  Above-threshold overlap candidates: 28

App-readiness
  Final embedding artifact saved.
  Final pairwise similarity artifact saved.
  Final top-match table saved.
  pgvector-ready export saved.

Notebook 05 is complete and the project is ready to move into the Streamlit demo app stage.
